In [ ]:
from scipy import signal
import numpy as np
import pandas as pd

# butterworth_filter

fs = 1000
bf = signal.butter(
    4,
    [20, 400],
    btype='bandpass',
    fs=fs,  
    output='sos'
)

#intialization of Global variables 

cols = ["Ch1", "Ch2", "Ch3", "Ch4", "Ch5", "Ch6", "Ch7", "Ch8"]
fs = 1000
window_size = 250     # 250 samples = 250 ms
step_size = 250       # No overlap  

# feature extraction functions 

def hjorth_features(signal):
    x = np.asarray(signal, dtype=float)
    dx = np.diff(x)
    ddx = np.diff(dx)
    activity = np.var(x)
    mobility = np.sqrt(np.var(dx) / activity) if activity > 0 else 0
    mobility_dx = (np.sqrt(np.var(ddx) / np.var(dx))
        if np.var(dx) > 0 else 0 )
    complexity = mobility_dx / mobility if mobility > 0 else 0
    return activity, mobility, complexity

def calculate_mav(signal):
    return np.mean(np.abs(signal))

def calculate_rms(signal):
    return np.sqrt(np.mean(signal ** 2))

def calculate_waveform_length(signal):
    return np.sum(np.abs(np.diff(signal)))

def calculate_ssc(signal):
    diff1 = np.diff(signal)
    return np.sum(diff1[:-1] * diff1[1:] < 0)  

def calculate_variance(signal):
    return np.var(signal)

def calculate_iemg(signal):
    return np.sum(np.abs(signal))    

def Zero_Crossing(signal, threshold=0.01):
    signal = np.asarray(signal)

    crossings = ( (signal[:-1] * signal[1:] < 0) &
        (np.abs(signal[1:] - signal[:-1]) >= threshold) )

    return int(np.sum(crossings))    
    

# begin in the loop for all subjects , with all levels , with all repetitions, for all channels , filtering first , normalizing then windowing 
# and then feature extraction using the defined functions in the begininng

for c in [1,2,3,4,5,6,7] :

    for s in [1,2,3,4,5,6,7,8,9,10]:
            
        for f in [20, 30, 40, 50, 60, 70, 80]:
            
            for n in range(1, 5):
                file_name = f"Downloads/ContractionIntensity-main/ContractionIntensity-main/S{s}/S{s}_{f}P_C{c}_R{n}.csv"
                
                df = pd.read_csv(file_name, names=cols)
    
                filtered_df = pd.DataFrame()    
        
                for i in range(1, 9):
                    original_channel = f"Ch{i}"
                    filtered_channel = f"Ch{i}"
                    signal_data = pd.to_numeric(df[original_channel], errors="coerce").to_numpy()
                    filtered_df[filtered_channel] = signal.sosfilt(bf, signal_data)
                    
                output_file = f"all_files/Filtered/S{s}_{f}P_C{c}_R{n}_filtered.csv"
                
                filtered_df.to_csv(output_file, index=False)
        
                file_path = f"all_files/Filtered/S{s}_{f}P_C{c}_R{n}_filtered.csv"
                
                df = pd.read_csv(file_path)
                
                #normalized_data = pd.DataFrame() 
                
                #for j in range(1, 9):
                    #normalized_data[f"Ch{j}"] = df[f"Ch{j}"] / mvc_data.loc[j, f"S{s}"]
                # use dierctly the normalized data 
                #df = normalized_data    
                features = []     
                window_number = 1
                channels = [f"Ch{i}" for i in range(1,9)]
                for start in range(0, len(df) - window_size + 1, step_size):
                    end = start + window_size
                    row = {"Window": window_number,
                        "Start_Sample": start,
                        "End_Sample": end - 1,
                        "MVC_Percent": f }
                    for channel in channels:
                        window = df[channel].iloc[start:end].to_numpy()
                        # Extract features
                        row[f"{channel}_MAV"] = calculate_mav(window)
                        row[f"{channel}_RMS"] = calculate_rms(window)
                        row[f"{channel}_WL"] = calculate_waveform_length(window)
                        row[f"{channel}_ssc"] = calculate_ssc(window)
                        row[f"{channel}_iemg"] = calculate_iemg(window)
                        row[f"{channel}_variance"] = calculate_variance(window)
                        (row[f"{channel}_Hjorth_variance"] , 
                        row[f"{channel}_Hjorth_mobility"], 
                        row[f"{channel}_Hjorth_complexity"],)=  hjorth_features(window)
                        row[f"{channel}_ZC"] = Zero_Crossing(window)
                        
                    features.append(row)
                
                    window_number += 1
                
                features_df = pd.DataFrame(features)
                
                output_file = f"all_files/Features/S{s}_{f}P_C{c}_R{n}_8Ch_features.csv"
                
                features_df.to_csv(output_file, index=False)
                
                print("\nSaved to:", output_file)

import pandas as pd
from pathlib import Path
folder = Path("all_files/Features")
for c in [1,2,3,4,5,6,7]:
    for s in [1,2,3,4,5,6,7,8,9,10]:     
        dataframes = []
        files = [
            folder / f"S{s}_50P_C{c}_R4_8Ch_features.csv",
            folder / f"S{s}_80P_C{c}_R2_8Ch_features.csv",
            folder / f"S{s}_70P_C{c}_R2_8Ch_features.csv",
            folder / f"S{s}_40P_C{c}_R2_8Ch_features.csv",
            folder / f"S{s}_60P_C{c}_R4_8Ch_features.csv",
            folder / f"S{s}_20P_C{c}_R4_8Ch_features.csv",
            folder / f"S{s}_80P_C{c}_R4_8Ch_features.csv",
            folder / f"S{s}_60P_C{c}_R2_8Ch_features.csv",
            folder / f"S{s}_50P_C{c}_R1_8Ch_features.csv",
            folder / f"S{s}_40P_C{c}_R4_8Ch_features.csv",
            folder / f"S{s}_70P_C{c}_R1_8Ch_features.csv",
            folder / f"S{s}_20P_C{c}_R2_8Ch_features.csv",
            folder / f"S{s}_50P_C{c}_R2_8Ch_features.csv",
            folder / f"S{s}_30P_C{c}_R4_8Ch_features.csv",
            folder / f"S{s}_80P_C{c}_R1_8Ch_features.csv",
            folder / f"S{s}_40P_C{c}_R1_8Ch_features.csv",
            folder / f"S{s}_60P_C{c}_R1_8Ch_features.csv",
            folder / f"S{s}_30P_C{c}_R2_8Ch_features.csv",
            folder / f"S{s}_70P_C{c}_R4_8Ch_features.csv",
            folder / f"S{s}_30P_C{c}_R1_8Ch_features.csv",
            folder / f"S{s}_20P_C{c}_R1_8Ch_features.csv",]
        
        for file in files:
            df = pd.read_csv(file)
            dataframes.append(df)
            # Combine all DataFrames
        data = pd.concat(dataframes, ignore_index=True)
        data.to_csv(f"all_files/Train/Dataset_S{s}_C{c}_features.csv",index=False)
        

for c in [1,2,3,4,5,6,7]:
    for s in [1,2,3,4,5,6,7,8,9,10]:
        dataframes = []
        files = [
            folder / f"S{s}_40P_C{c}_R3_8Ch_features.csv",
            folder / f"S{s}_50P_C{c}_R3_8Ch_features.csv",
            folder / f"S{s}_60P_C{c}_R3_8Ch_features.csv",
            folder / f"S{s}_70P_C{c}_R3_8Ch_features.csv",
            folder / f"S{s}_80P_C{c}_R3_8Ch_features.csv",
            folder / f"S{s}_20P_C{c}_R3_8Ch_features.csv",
            folder / f"S{s}_30P_C{c}_R3_8Ch_features.csv",]
        
        for file in files:
            df = pd.read_csv(file)
            dataframes.append(df)
        data_test = pd.concat(dataframes, ignore_index=True)
        output_file = f"all_files/Test/Test_Dataset_S{s}_C{c}_features.csv"
        data_test.to_csv(output_file,index=False)
        print ("/n saved to :",output_file)